In [ ]:
import pandas as pd
import numpy as np

## 2. Descarga de los TOIs y filtro de multiplicidad

Se consulta el servicio **TAP** del NASA Exoplanet Archive con una consulta tipo SQL escrita dentro de la URL (los `+` reemplazan los espacios).

| Columna | Significado |
|---|---|
| `toi` | Nombre del candidato (parte entera = sistema, decimal = planeta) |
| `tid` | TIC ID, identificador de la **estrella** |
| `tfopwg_disp` | Disposición: CP, PC, KP, FP, FA |
| `pl_orbper` | Período orbital (días) |
| `pl_rade` | Radio del planeta (R⊕) |
| `st_tmag` | Magnitud en la banda de TESS |
| `st_teff` | Temperatura efectiva de la estrella (K) |
| `st_rad` | Radio estelar (R☉) |
| `ra`, `dec` | Coordenadas (grados) |

**Filtro de multiplicidad:** `groupby("tid")["toi"].transform("count")` cuenta cuántos candidatos tiene cada estrella y le asigna ese número a cada fila; se conservan las estrellas con más de un candidato.

> Este filtro selecciona estrellas donde TESS detectó **más de un planeta en tránsito**. Es intencional: así se puede medir el radio (y luego la densidad) de más de un planeta del mismo sistema.

In [ ]:
url = ("https://exoplanetarchive.ipac.caltech.edu/TAP/sync?query="
       "select+toi,tid,tfopwg_disp,pl_orbper,pl_rade,st_tmag,st_teff,st_rad,ra,dec+from+toi"
       "&format=csv")
toi = pd.read_csv(url)
multi = toi[toi.groupby("tid")["toi"].transform("count") > 1]
print(len(multi), "candidatos en sistemas múltiples")

576 candidatos en sistemas múltiples


### Vista previa de la tabla

In [ ]:
multi.head()

,toi,tid,tfopwg_disp,pl_orbper,pl_rade,st_tmag,st_teff,st_rad,ra,dec
36,1064.01,79748331,CP,6.443866,2.460344,10.0059,4803.0,0.737189,296.003935,-47.562030
37,1064.02,79748331,CP,12.226560,2.601477,10.0059,4803.0,0.737189,296.003935,-47.562030
79,1027.01,20318757,PC,3.283456,2.876480,10.2103,4272.0,0.680000,167.133275,-29.653135
80,1027.02,20318757,PC,11.028750,2.765860,10.2103,4272.0,0.680000,167.133275,-29.653135
81,1027.03,20318757,PC,5.011329,2.523140,10.2103,4272.0,0.680000,167.133275,-29.653135


## 3. Filtros para seleccionar enanas M

### 3.1 Temperatura efectiva: $T_{\rm eff} < 3800$ K
Se imprime también cuántos valores faltan (`NaN`); las comparaciones con `NaN` dan `False`, así que esas filas quedan fuera.

In [ ]:
mask_teff = multi['st_teff'] < 3800
print(multi['st_teff'].isna().sum())
mask_teff.sum()

2


np.int64(84)

### 3.2 Radio estelar: $0.08 \le R_\star \le 0.6\,R_\odot$

In [ ]:
mask_rad = multi['st_rad'].between(0.08, 0.6)
print(multi['st_rad'].isna().sum())
print(mask_rad.sum())

11
102


### 3.3 Brillo: $T_{\rm mag} < 12$

Las enanas M emiten principalmente en el rojo/infrarrojo, por lo que su $T_{\rm mag}$ es mucho más brillante que su magnitud $V$. Un corte de $T_{\rm mag} < 12$ conserva más sistemas múltiples que $T_{\rm mag} < 11$ (24 sistemas frente a 9) sin perder estrellas útiles para espectrografía.

La precisión en VR está limitada por ruido de fotones: $\sigma_{\rm VR} \propto 1/\sqrt{N_{\rm fotones}}$, de modo que cada magnitud más débil aumenta el error en un factor $\sqrt{2.512}\approx 1.6$.

In [ ]:
TMAG_MAX = 12
mask_mag = multi['st_tmag'] < TMAG_MAX
mask_mag.sum()

np.int64(515)

### 3.4 Combinación de los tres criterios

In [ ]:
mask_EnanasM = mask_rad & mask_teff & mask_mag
mask_EnanasM.sum()

np.int64(58)

In [ ]:
Enanas_M = multi[mask_EnanasM].copy()
len(Enanas_M)

58

### 3.5 Prueba de consistencia entre criterios
- `Prueba`: estrellas con $T_{\rm eff} < 3800$ K pero radio fuera del rango → **0**: toda estrella fría es también pequeña.
- `Prueba2`: estrellas pequeñas pero con $T_{\rm eff} \ge 3800$ K → **18**: probablemente enanas K tardías / M tempranas en el borde del corte.

In [ ]:
Prueba = ~mask_rad & mask_teff
print(Prueba.sum())
Prueba2 = ~mask_teff & mask_rad
print(Prueba2.sum())

0
18


## 4. Ordenar por brillo

In [ ]:
Enanas_M_ordenadas = Enanas_M.sort_values('st_tmag', ascending=True)
Enanas_M_ordenadas.head()

,toi,tid,tfopwg_disp,pl_orbper,pl_rade,st_tmag,st_teff,st_rad,ra,dec
875,175.02,307210830,CP,7.450730,3.045280,9.41242,3469.0,0.31000,124.532860,-68.314466
876,175.03,307210830,CP,2.253106,0.826635,9.41242,3469.0,0.31416,124.532860,-68.314466
874,175.01,307210830,CP,3.690676,1.367510,9.41242,3469.0,0.31000,124.532860,-68.314466
844,1730.01,318022259,CP,6.226231,2.755240,10.35950,3691.0,0.53000,107.865826,48.327933
846,1730.03,318022259,CP,12.573957,2.435900,10.35950,3691.0,0.53000,107.865826,48.327933


## 5. Disposición de los candidatos (TFOPWG)

| Código | Significado |
|---|---|
| CP | Planeta confirmado |
| PC | Planeta candidato |
| KP | Planeta conocido antes de TESS |
| FP | Falso positivo |
| FA | Falsa alarma |

In [ ]:
Enanas_M_ordenadas['tfopwg_disp'].value_counts(dropna=False)


,count
tfopwg_disp,
CP,35
PC,21
KP,2


### 5.1 Planetas conocidos (KP)
TOI-256 corresponde al sistema **LHS 1140** (dos planetas ya conocidos y bien estudiados).

In [ ]:
Enanas_M_ordenadas[Enanas_M_ordenadas['tfopwg_disp'] == 'KP']

,toi,tid,tfopwg_disp,pl_orbper,pl_rade,st_tmag,st_teff,st_rad,ra,dec,instrumentos
1567,256.02,92226327,KP,3.777942,1.565994,11.2991,3131.0,0.215535,11.248632,-15.274109,Ambos
1531,256.01,92226327,KP,24.737341,1.843945,11.2991,3131.0,0.215535,11.248632,-15.274109,Ambos


### 5.2 Eliminación de falsos positivos y falsas alarmas

In [ ]:
mask_tipo = ~Enanas_M_ordenadas['tfopwg_disp'].isin(['FP', 'FA'])
Enanas_M_limpias = Enanas_M_ordenadas[mask_tipo].copy()
print(len(Enanas_M_ordenadas))

58


## 6. Conteo de sistemas y verificación de multiplicidad
Se verifica que, tras los filtros, cada estrella conserve al menos dos candidatos (los parámetros estelares pueden variar entre filas de un mismo sistema).

In [ ]:
print(Enanas_M_ordenadas['tid'].nunique())
print(Enanas_M_ordenadas['tid'].value_counts())

24
tid
150428135    4
307210830    3
77156829     3
318022259    3
54962195     3
300710077    3
235678745    3
259377017    3
271596225    3
224298134    2
270355392    2
36724087     2
243185500    2
153065527    2
149788158    2
467179528    2
329148988    2
415969908    2
92226327     2
219195044    2
468983280    2
321669174    2
299798795    2
232650365    2
Name: count, dtype: int64


### 6.1 Ejemplo: el único sistema de 4 planetas (TOI-700)

In [ ]:
Enanas_M_ordenadas[Enanas_M_ordenadas['tid'] == 150428135]

,toi,tid,tfopwg_disp,pl_orbper,pl_rade,st_tmag,st_teff,st_rad,ra,dec
6732,700.02,150428135,CP,37.423684,1.329657,10.9102,3482.0,0.419457,97.095717,-65.578615
6731,700.01,150428135,CP,16.051106,2.586921,10.9102,3482.0,0.419457,97.095717,-65.578615
6733,700.03,150428135,CP,9.977273,0.961729,10.9102,3482.0,0.419457,97.095717,-65.578615
7375,700.04,150428135,CP,27.809690,0.949186,10.9102,3482.0,0.419457,97.095717,-65.578615


## 7. Instrumentos disponibles según la declinación

Criterio aproximado:
- $\delta < -30^\circ$ → **Sur**: HARPS, ESPRESSO, NIRPS (La Silla / Paranal, Chile)
- $\delta > +30^\circ$ → **Norte**: SPIRou (CFHT), MAROON-X (Gemini Norte)
- $-30^\circ \le \delta \le +30^\circ$ → **Ambos** hemisferios

In [ ]:
Enanas_M_ordenadas['instrumentos'] = np.where(Enanas_M_ordenadas['dec'] < -30, 'Sur: HARPS, ESPRESSO, NIRPS', np.where(Enanas_M_ordenadas['dec'] > 30, 'Norte: SPIRou, MAROON-X', 'Ambos'))
Enanas_M_ordenadas[['toi', 'dec', 'instrumentos']]

,toi,dec,instrumentos
875,175.02,-68.314466,"Sur: HARPS, ESPRESSO, NIRPS"
876,175.03,-68.314466,"Sur: HARPS, ESPRESSO, NIRPS"
874,175.01,-68.314466,"Sur: HARPS, ESPRESSO, NIRPS"
844,1730.01,48.327933,"Norte: SPIRou, MAROON-X"
846,1730.03,48.327933,"Norte: SPIRou, MAROON-X"
845,1730.02,48.327933,"Norte: SPIRou, MAROON-X"
7595,696.03,-39.790874,"Sur: HARPS, ESPRESSO, NIRPS"
7594,696.02,-39.790874,"Sur: HARPS, ESPRESSO, NIRPS"
7593,696.01,-39.790874,"Sur: HARPS, ESPRESSO, NIRPS"
1549,270.03,-51.957394,"Sur: HARPS, ESPRESSO, NIRPS"


### 7.1 Número de sistemas por hemisferio

In [ ]:
Num_sistemas = Enanas_M_ordenadas.drop_duplicates('tid')
print(len(Num_sistemas))
Num_sistemas['instrumentos'].value_counts()

24


,count
instrumentos,
"Sur: HARPS, ESPRESSO, NIRPS",9
Ambos,8
"Norte: SPIRou, MAROON-X",7


## 8. Exportar la tabla de candidatos

In [ ]:
Enanas_M_ordenadas.to_csv('candidatos_enanasM_multiples.csv', index=False)

## 9. Discusión preliminar: ¿qué sistemas son medibles en VR?

La semiamplitud esperada de la VR (órbita circular, $\sin i \approx 1$ por transitar):

$$K \approx 28.4\ {\rm m\,s^{-1}}\left(\frac{M_p}{M_J}\right)\left(\frac{P}{1\ {\rm año}}\right)^{-1/3}\left(\frac{M_\star}{M_\odot}\right)^{-2/3}$$

Masas estimadas a partir del radio: $M \approx R^3$ para $R < 1.5\,R_\oplus$ y $M \approx 2.69\,R^{0.93}$ (Weiss & Marcy 2014) para $1.5 < R < 4\,R_\oplus$; para enanas M se toma $M_\star \approx R_\star$.

| Sistema | Planetas | $K$ esperado | Comentario |
|---|---|---|---|
| TOI-700 | 4 | 0.3 – 2.9 m/s | Tres planetas con $K < 1$ m/s → masas no medibles con datos de archivo; **descartado para densidades** |
| TOI-256 (LHS 1140) | 2 | ~2.4 y ~3.8 m/s (radios de literatura) | Medible, muy estudiado, observable desde ambos hemisferios → **candidato principal** |

> **Nota:** los radios de la tabla de TOIs pueden diferir de los publicados (LHS 1140: 1.84 y 1.57 R⊕ en TOI vs ~1.73 y ~1.27 R⊕ en literatura). Como $\rho \propto M/R^3$, para las densidades se deben usar los radios del mejor ajuste de tránsito.